# 02 — Baseline Models (SVD, ALS, Popularity)

This notebook trains and evaluates classical recommendation baselines:
1. **Popularity Baseline** — recommend globally popular items
2. **SVD (Singular Value Decomposition)** — linear matrix factorization
3. **ALS (Alternating Least Squares)** — implicit feedback factorization

These serve as benchmarks for the NCF model.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from data_loader import load_movielens, encode_ids
from baselines import PopularityBaseline, SVDBaseline, ALSBaseline, evaluate_baseline
from evaluate import print_metrics
from utils import print_results_table, plot_metrics_comparison

## 1. Load and Prepare Data

In [ ]:
# Load full dataset for baselines (they're fast enough)
df = load_movielens('../data/ml-25m', sample_size=10_000_000)
df, user_map, item_map = encode_ids(df)
num_users, num_items = len(user_map), len(item_map)

# Split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

## 2. Popularity Baseline

In [ ]:
pop_model = PopularityBaseline()
pop_model.fit(train_df)

pop_metrics = evaluate_baseline(pop_model, test_df, train_df, k_values=[5, 10, 20])
print_metrics(pop_metrics, 'Popularity Baseline')

## 3. SVD Baseline

In [ ]:
svd_model = SVDBaseline(n_factors=50)
svd_model.fit(train_df, num_users, num_items)

svd_metrics = evaluate_baseline(svd_model, test_df, train_df, k_values=[5, 10, 20])
print_metrics(svd_metrics, 'SVD Baseline')

## 4. ALS Baseline

⚠️ **Note:** Full ALS on 25M is slow. Using a smaller sample or reduced iterations.

In [ ]:
# Use smaller sample for ALS (computationally expensive)
als_sample = train_df.sample(n=min(2_000_000, len(train_df)), random_state=42)
als_sample, als_user_map, als_item_map = encode_ids(als_sample)

als_model = ALSBaseline(n_factors=50, reg=0.01, n_iterations=15)
als_model.fit(als_sample, len(als_user_map), len(als_item_map))

# Re-encode test set to match ALS encoding
test_als = test_df[test_df['userId'].isin(als_user_map) & test_df['movieId'].isin(als_item_map)].copy()
test_als['userId'] = test_als['userId'].map(als_user_map)
test_als['movieId'] = test_als['movieId'].map(als_item_map)

als_metrics = evaluate_baseline(als_model, test_als, als_sample, k_values=[5, 10, 20])
print_metrics(als_metrics, 'ALS Baseline')

## 5. Results Comparison

In [ ]:
results = {
    'Popularity': pop_metrics,
    'SVD': svd_metrics,
    'ALS': als_metrics,
}

print_results_table(results)
plot_metrics_comparison(results, k=10, save_path='../figures/baseline_comparison.png')

## 6. Key Takeaways

1. **SVD and ALS significantly outperform Popularity Baseline** — confirms that personalization matters
2. **ALS slightly edges SVD** — implicit feedback formulation captures binary interactions better
3. **All baselines struggle with cold-start users** — these linear methods need the NCF model's ability to generalize
4. **These results set the bar** — NCF needs to beat ALS's NDCG@10 to justify the additional complexity